In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import faiss
import gc
import pickle
import time
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# ============================================================
# MEMORY-SAFE CONFIGURATION (< 3 GB RAM PEAK)
# ============================================================
K = 40                        # Top-K candidates
MIN_SIMILARITY = 0.05         # Threshold
TFIDF_VOCAB = 25_000          # Large vocabulary for rich n-grams
EMBED_DIM = 128               # Compressed to 128 dims (512 MB max index size!)
CHUNK_SIZE_SEARCH = 20_000    # Safe query batch size

DATA_DIR = "../../dataset/cleaned"
OUTPUT_DIR = "."

print("Lean config loaded! Max FAISS index RAM will be ~500 MB.")

In [ ]:
print("Loading TEST parquet files...")
t0 = time.time()

s1_test  = pl.read_parquet(f"{DATA_DIR}/test_source1.parquet")
s2_test  = pl.read_parquet(f"{DATA_DIR}/test_source2.parquet")
s3_test  = pl.read_parquet(f"{DATA_DIR}/test_source3.parquet")

s2s3_test = pl.concat([s2_test, s3_test])
del s2_test, s3_test
gc.collect()

print(f"Loaded in {time.time() - t0:.2f}s!")
print(f"  s1_test:   {len(s1_test):,} rows")
print(f"  s2s3_test: {len(s2s3_test):,} rows")

In [ ]:
print("Loading TEST parquet files...")
t0 = time.time()

s1_test  = pl.read_parquet(f"{DATA_DIR}/test_source1.parquet")
s2_test  = pl.read_parquet(f"{DATA_DIR}/test_source2.parquet")
s3_test  = pl.read_parquet(f"{DATA_DIR}/test_source3.parquet")

s2s3_test = pl.concat([s2_test, s3_test])
del s2_test, s3_test
gc.collect()

print(f"Loaded in {time.time() - t0:.2f}s!")
print(f"  s1_test:   {len(s1_test):,} rows")
print(f"  s2s3_test: {len(s2s3_test):,} rows")

In [ ]:
def prepare_combined_field(df):
    df = df.with_columns([
        pl.col("name_core").fill_null("").cast(pl.Utf8).alias("nc"),
        pl.col("name_phon").fill_null("").cast(pl.Utf8).alias("np"),
        pl.col("addr_clean").fill_null("").cast(pl.Utf8).alias("ac"),
        pl.col("addr_nums").fill_null("").cast(pl.Utf8).alias("an"),
        pl.col("country").fill_null("").cast(pl.Utf8).str.to_lowercase().str.strip_chars().alias("country_clean"),
        pl.col("state").fill_null("").cast(pl.Utf8).str.strip_chars().alias("state_clean"),
    ])

    df = df.with_columns(
        (pl.col("nc") + " " + pl.col("nc") + " " + pl.col("nc") + " " +
         pl.col("np") + " " +
         pl.col("ac") + " " +
         pl.col("an") + " " + pl.col("an"))
        .str.replace_all(r"\s+", " ")
        .str.strip_chars()
        .alias("combined_text")
    )
    return df

print("Building search strings for TEST...")
s1_test   = prepare_combined_field(s1_test)
s2s3_test = prepare_combined_field(s2s3_test)
print("Done!")

In [ ]:
def block_partition(s1_ids, s1_texts, s2s3_ids, s2s3_texts, partition_label):
    n_s1 = len(s1_ids)
    n_s2s3 = len(s2s3_ids)
    
    if n_s1 == 0 or n_s2s3 == 0:
        return {}
    
    print(f"    S1: {n_s1:,} | S2+S3: {n_s2s3:,}", end=" | ", flush=True)
    
    # 1. TF-IDF on char n-grams
    all_texts = s1_texts + s2s3_texts
    n_total = len(all_texts)
    
    # Adaptive vocab & embedding dimension for small partitions
    cur_vocab = min(TFIDF_VOCAB, max(1000, n_total * 2))
    cur_dim = min(EMBED_DIM, max(32, n_total // 10))
    
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        max_features=cur_vocab,
        sublinear_tf=True,
        dtype=np.float32,
    )
    
    tfidf_sparse = vectorizer.fit_transform(all_texts)
    del all_texts
    
    # 2. Dimensionality Reduction via SVD (LSA) -> Compress to 128 dims
    if tfidf_sparse.shape[1] > cur_dim and n_total > cur_dim:
        svd = TruncatedSVD(n_components=cur_dim, algorithm="randomized", n_iter=2, random_state=42)
        dense_vecs = svd.fit_transform(tfidf_sparse).astype(np.float32)
    else:
        dense_vecs = tfidf_sparse.toarray().astype(np.float32)
        cur_dim = dense_vecs.shape[1]
    
    del tfidf_sparse, vectorizer
    gc.collect()
    
    s1_dense = dense_vecs[:n_s1]
    s2s3_dense = dense_vecs[n_s1:]
    del dense_vecs
    
    # 3. L2 Normalize for Cosine Similarity
    faiss.normalize_L2(s1_dense)
    faiss.normalize_L2(s2s3_dense)
    
    # 4. Exact FAISS FlatIP (Ultra-fast & lightweight at 128 dims)
    index = faiss.IndexFlatIP(cur_dim)
    index.add(s2s3_dense)
    del s2s3_dense
    gc.collect()
    
    # 5. Search in batches
    results = {}
    for start in range(0, n_s1, CHUNK_SIZE_SEARCH):
        end = min(start + CHUNK_SIZE_SEARCH, n_s1)
        sub_s1 = s1_dense[start:end]
        
        distances, indices = index.search(sub_s1, K)
        
        for i in range(end - start):
            s1_id = s1_ids[start + i]
            cands = []
            for j in range(K):
                idx = indices[i][j]
                score = float(distances[i][j])
                if idx >= 0 and score >= MIN_SIMILARITY:
                    cands.append((s2s3_ids[idx], score))
            results[s1_id] = cands
            
    del s1_dense, index
    gc.collect()
    
    total_cands = sum(len(v) for v in results.values())
    print(f"pairs: {total_cands:,}")
    return results

print("SVD-FAISS Engine compiled successfully!")

In [ ]:
def run_dataset_blocking(s1_pl, s2s3_pl, dataset_name):
    """
    Blocks per (country, state) partition using native Polars extraction.
    """
    print(f"\n{'=' * 70}")
    print(f"BLOCKING: {dataset_name.upper()} SET")
    print(f"{'=' * 70}")
    
    # Get all unique (country, state) pairs without pandas
    partitions = (
        s1_pl
        .select(["country_clean", "state_clean"])
        .unique()
        .sort(["country_clean", "state_clean"])
    )
    
    countries = partitions["country_clean"].to_list()
    states = partitions["state_clean"].to_list()
    
    print(f"Total partitions (country+state): {len(countries)}")
    
    all_results = {}
    
    for country, state in zip(countries, states):
        label = f"{country}/{state}"
        
        # Filter partitions
        s1_c = s1_pl.filter(
            (pl.col("country_clean") == country) & 
            (pl.col("state_clean") == state)
        )
        s2s3_c = s2s3_pl.filter(
            (pl.col("country_clean") == country) & 
            (pl.col("state_clean") == state)
        )
        
        n_s1 = len(s1_c)
        n_s2s3 = len(s2s3_c)
        
        if n_s1 == 0 or n_s2s3 == 0:
            if n_s1 > 0:
                for eid in s1_c["entity_id"].to_list():
                    all_results[eid] = []
            continue
        
        if n_s1 > 100:
            print(f"\n  [{label}]")
        
        # Native python lists extraction (super fast & zero dependency)
        s1_ids = s1_c["entity_id"].to_list()
        s1_texts = s1_c["combined_text"].to_list()
        s2s3_ids = s2s3_c["entity_id"].to_list()
        s2s3_texts = s2s3_c["combined_text"].to_list()
        
        partition_results = block_partition(s1_ids, s1_texts, s2s3_ids, s2s3_texts, label)
        all_results.update(partition_results)
        
        del s1_ids, s1_texts, s2s3_ids, s2s3_texts, s1_c, s2s3_c
        gc.collect()
    
    # Ensure every single S1 entity exists in dictionary
    for eid in s1_pl["entity_id"].to_list():
        if eid not in all_results:
            all_results[eid] = []
    
    return all_results

print("Updated runner defined!")

In [ ]:
t0 = time.time()
test_results = run_dataset_blocking(s1_test, s2s3_test, "test")
print(f"\n TEST FINISHED in {(time.time() - t0)/60:.2f} minutes!")

# Save immediately
with open(f"{OUTPUT_DIR}/blocking_person_a_test.pkl", "wb") as f:
    pickle.dump(test_results, f)

print("Saved: blocking_person_a_test.pkl")

In [ ]:
# Free test memory completely
del s1_test, s2s3_test, test_results
gc.collect()

print("Loading TRAIN parquet files...")
s1_train  = pl.read_parquet(f"{DATA_DIR}/train_source1.parquet")
s2_train  = pl.read_parquet(f"{DATA_DIR}/train_source2.parquet")
s3_train  = pl.read_parquet(f"{DATA_DIR}/train_source3.parquet")

s2s3_train = pl.concat([s2_train, s3_train])
del s2_train, s3_train
gc.collect()

s1_train   = prepare_combined_field(s1_train)
s2s3_train = prepare_combined_field(s2s3_train)

# Run train blocking
t0 = time.time()
train_results = run_dataset_blocking(s1_train, s2s3_train, "train")
print(f"\n TRAIN FINISHED in {(time.time() - t0)/60:.2f} minutes!")

# Save train results
with open(f"{OUTPUT_DIR}/blocking_person_a_train.pkl", "wb") as f:
    pickle.dump(train_results, f)

print("Saved: blocking_person_a_train.pkl")

In [ ]:
import pickle
import numpy as np
import polars as pl
import os
import gc

OUTPUT_DIR = "."
GROUND_TRUTH_PATH = "../../dataset/train/train_ground_truth.tsv"

# ============================================================
# 1. EVALUATE TRAIN RECALL & STATS (LIGHTWEIGHT)
# ============================================================
print("=" * 60)
print("1. EVALUATING TRAIN BLOCKING & RECALL")
print("=" * 60)

train_path = f"{OUTPUT_DIR}/blocking_person_a_train.pkl"
print(f"Loading Train candidates from: {train_path} ...")
with open(train_path, "rb") as f:
    train_results = pickle.load(f)

n_entities_train = len(train_results)
total_pairs_train = 0
empty_train = 0
sum_score_train = 0.0
min_score_train = 1.0
max_score_train = 0.0

for s1_id, cands in train_results.items():
    n_c = len(cands)
    total_pairs_train += n_c
    if n_c == 0:
        empty_train += 1
    for _, score in cands:
        sum_score_train += score
        if score < min_score_train: min_score_train = score
        if score > max_score_train: max_score_train = score

avg_score_train = (sum_score_train / total_pairs_train) if total_pairs_train > 0 else 0

print(f"\n[TRAIN STATS]:")
print(f"  S1 Entities:         {n_entities_train:,}")
print(f"  Total Pairs:         {total_pairs_train:,}")
print(f"  Avg Cands / S1:      {total_pairs_train / n_entities_train:.1f}")
print(f"  Zero-Cand Entities:  {empty_train:,} ({100*empty_train/n_entities_train:.2f}%)")
print(f"  Score Range:         {min_score_train:.3f} to {max_score_train:.3f}")
print(f"  Avg Score:           {avg_score_train:.3f}")

# --- Ground Truth Recall Evaluation ---
print("\nEvaluating against Ground Truth...")
gt_path = GROUND_TRUTH_PATH
if not os.path.exists(gt_path):
    gt_path = "../../dataset/cleaned/train_ground_truth.tsv"

gt_train = pl.read_csv(gt_path, separator="\t")

total_true = 0
total_recovered = 0
per_entity_recall = []
n_singletons = 0

for row in gt_train.iter_rows(named=True):
    s1_id = row["source1_entity_id"]
    m_str = row["matched_entity_ids"]
    
    if s1_id not in train_results:
        continue
        
    if m_str is None or str(m_str).strip() == "":
        n_singletons += 1
        continue
        
    true_set = set(str(m_str).split(","))
    cand_ids = {cand[0] for cand in train_results[s1_id]}
    rec = len(true_set & cand_ids)
    
    total_true += len(true_set)
    total_recovered += rec
    per_entity_recall.append(rec / len(true_set))

micro_recall = total_recovered / total_true if total_true > 0 else 0
macro_recall = float(np.mean(per_entity_recall)) if per_entity_recall else 0

print("\n" + "=" * 50)
print("🎯 BLOCKING RECALL RESULTS (TRAIN)")
print("=" * 50)
print(f"  Non-Singletons Evaluated: {len(per_entity_recall):,}")
print(f"  Singletons (Skipped):     {n_singletons:,}")
print(f"  True Matches in GT:       {total_true:,}")
print(f"  Matches Captured in Top-K:{total_recovered:,}")
print(f"  MICRO RECALL:             {micro_recall:.4f}")
print(f"  MACRO RECALL:             {macro_recall:.4f}")
print("=" * 50)

# Free Train Results completely before touching Test!
del train_results, gt_train, per_entity_recall
gc.collect()

# ============================================================
# 2. EVALUATE TEST STATS (LIGHTWEIGHT)
# ============================================================
print("\n" + "=" * 60)
print("2. EVALUATING TEST BLOCKING STATS")
print("=" * 60)

test_path = f"{OUTPUT_DIR}/blocking_person_a_test.pkl"
print(f"Loading Test candidates from: {test_path} ...")
with open(test_path, "rb") as f:
    test_results = pickle.load(f)

n_entities_test = len(test_results)
total_pairs_test = 0
empty_test = 0
sum_score_test = 0.0
min_score_test = 1.0
max_score_test = 0.0

for s1_id, cands in test_results.items():
    n_c = len(cands)
    total_pairs_test += n_c
    if n_c == 0:
        empty_test += 1
    for _, score in cands:
        sum_score_test += score
        if score < min_score_test: min_score_test = score
        if score > max_score_test: max_score_test = score

avg_score_test = (sum_score_test / total_pairs_test) if total_pairs_test > 0 else 0

print(f"\n[TEST STATS]:")
print(f"  S1 Entities:         {n_entities_test:,}")
print(f"  Total Pairs:         {total_pairs_test:,}")
print(f"  Avg Cands / S1:      {total_pairs_test / n_entities_test:.1f}")
print(f"  Zero-Cand Entities:  {empty_test:,} ({100*empty_test/n_entities_test:.2f}%)")
print(f"  Score Range:         {min_score_test:.3f} to {max_score_test:.3f}")
print(f"  Avg Score:           {avg_score_test:.3f}")

del test_results
gc.collect()

print("\n Evaluation Complete with ZERO Memory Leaks!")